# Role Generation for Spider Database Tables

This notebook performs role-based access control (RBAC) analysis for the Spider database collection using LLM.

In [1]:
import sys
from pathlib import Path

# Add project root to path
project_root = Path('/home/feiy/Role-SQL-benchmark')
sys.path.append(str(project_root))

# Import required modules
from src.role_parser import RoleGenerator, ParallelRoleGenerator
from src.utils.sql_data_process import SpiderDataProcessor
from dotenv import load_dotenv
import os
import json
from datetime import datetime
import logging
import random
import importlib
import src.llm_oracle as oracle

# Setup output directories
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
log_dir = project_root / 'logs'
output_dir = project_root / 'outputs'

for directory in [log_dir, output_dir]:
    directory.mkdir(exist_ok=True)

# Configure logging
log_file = log_dir / f'role_assignment_{timestamp}.log'
logging.getLogger().handlers.clear()

logger = logging.getLogger('role_assignment')
logger.setLevel(logging.INFO)
logger.handlers.clear()

file_handler = logging.FileHandler(str(log_file))
console_handler = logging.StreamHandler(sys.stdout)
formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')

for handler in [file_handler, console_handler]:
    handler.setFormatter(formatter)
    logger.addHandler(handler)

logger.propagate = False
logger.info(f"Starting new session at {timestamp}")
logger.info(f"Log file: {log_file}")
logger.info(f"Output directory: {output_dir}")

2025-09-17 11:14:09,909 - INFO - Starting new session at 20250917_111409
2025-09-17 11:14:09,910 - INFO - Log file: /home/feiy/Role-SQL-benchmark/logs/role_assignment_20250917_111409.log
2025-09-17 11:14:09,910 - INFO - Output directory: /home/feiy/Role-SQL-benchmark/outputs
2025-09-17 11:14:09,910 - INFO - Log file: /home/feiy/Role-SQL-benchmark/logs/role_assignment_20250917_111409.log
2025-09-17 11:14:09,910 - INFO - Output directory: /home/feiy/Role-SQL-benchmark/outputs


/home/feiy/anaconda3/envs/llm4db/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Load environment variables and API keys
load_dotenv()
importlib.reload(oracle)

In [ ]:
# DeepSeek demo
DEEPSEEK_API_KEY = os.getenv('DEEPSEEK_API_KEY')
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

if not DEEPSEEK_API_KEY:
    print("Warning: Cannot find DEEPSEEK_API_KEY in environment")
    print("Please set it in the .env file or environment")
if not OPENAI_API_KEY:
    print("Warning: Cannot find OPENAI_API_KEY in environment")
    print("Please set it in the .env file or environment")


### 1. Setup LLM Oracle instance

In [ ]:
def create_oracle(model_name, api_key):
    return oracle.Oracle(model_name, api_key)

#### 1.1. Deepseek test

##### 1.1(a) deepseek demo test

In [ ]:
# if not DEEPSEEK_API_KEY:
#     print("Warning: DEEPSEEK_API_KEY not found in environment variables")
#     print("Please set it in your .env file or environment")
# else:
#     # Create Oracle instance with DeepSeek model
#     oracle = Oracle(model="deepseek-chat", apikey=DEEPSEEK_API_KEY)
    
#     # Test the model
#     test_response = oracle.query(
#         prompt_sys="You are a helpful assistant.",
#         prompt_user="Say Hi.",
#         temp=0.7,
#         top_p=0.9
#     )
    
#     print("Model Response:")
#     print("="*50)
#     print(test_response['answer'])

##### 1.1(b) prompt caching test

In [ ]:
# # Create two identical requests to test caching
# prompt_sys = """You are a helpful assistant. You should:
# 1. Be concise and clear in your responses
# 2. Always strive to provide accurate information
# 3. Maintain a professional and friendly tone
# 4. Use appropriate formatting when needed
# 5. Ask for clarification if something is unclear"""

# prompt_user = """Please introduce yourself and tell me about your capabilities.
# Make sure to mention:
# 1. Your name
# 2. Your main areas of expertise
# 3. How you can help users
# 4. Any limitations users should be aware of"""

In [ ]:
# def make_query(prompt_sys, prompt_user):
#     """Execute query and return response"""
#     response = oracle.query(
#         prompt_sys=prompt_sys,
#         prompt_user=prompt_user,
#     )
#     # if response.get('answer'):
#     #     print(f"Response: {response['answer']}")
#     return response

# def print_cache_stats(response, label=None):
#     """Print cache statistics for a response"""
#     if label:
#         print(f"\n=== {label} Statistics ===")
    
#     usage = response.get('usage', {})
#     prompt_details = usage.get('prompt_tokens_details', None)
    
#     print("\nToken Statistics:")
#     print(f"  Prompt tokens: {usage.get('prompt_tokens', 0)}")
#     print(f"  Completion tokens: {usage.get('completion_tokens', 0)}")
#     print(f"  Total tokens: {usage.get('total_tokens', 0)}")
    
#     print("\nCache Statistics:")
#     if prompt_details:
#         if isinstance(prompt_details, str):
#             print(f"  Raw info: {prompt_details}")
#         else:
#             cached = getattr(prompt_details, 'cached_tokens', 0)
#             non_cached = usage.get('prompt_tokens', 0) - cached
#             print(f"  Cached tokens: {cached}")
#             print(f"  Non-cached tokens: {non_cached}")
#             if cached > 0:
#                 print(f"  Cache hit rate: {(cached / usage.get('prompt_tokens', 1)) * 100:.1f}%")

In [ ]:
# response1 = make_query(prompt_sys, prompt_user)
# response2 = make_query(prompt_sys, prompt_user)
# response3 = make_query(prompt_sys, prompt_user)

# print_cache_stats(response1, "First Call")
# print_cache_stats(response2, "Second Call")
# print_cache_stats(response3, "Third Call")

#### 1.2. OpenAI Test

In [ ]:
# Example: create Oracle instance (model and api_key should be set according to your environment)

# MODEL_NAME = 'gpt-4o'  # or any supported model
# create_oracle_instance = create_oracle(model_name=MODEL_NAME, api_key=OPENAI_API_KEY)
# print(f"Oracle instance created for model: {MODEL_NAME}")

In [ ]:
# build a simple prompt for testing

# prompt_sys = """You are a helpful assistant. You should:
# 1. Be concise and clear in your responses
# 2. Always strive to provide accurate information
# 3. Maintain a professional and friendly tone
# 4. Use appropriate formatting when needed
# 5. Ask for clarification if something is unclear"""
# prompt_user = """Please introduce yourself and tell me about your capabilities.
# Make sure to mention:
# 1. Your name
# 2. Your main areas of expertise
# 3. How you can help users
# 4. Any limitations users should be aware of"""
# response = openai_oracle_instance.query(
#     prompt_sys=prompt_sys,
#     prompt_user=prompt_user,
# )
# print(f"Response: {response['answer']}")

### 2. Role Assignment for Spider Database Tables

This section aims to:
1. Read schema information from Spider database
2. Use LLM to generate appropriate roles for each table
3. Test the role assignment with sample cases

In [2]:
# Setup Spider database path
SPIDER_ROOT = project_root / 'data/spider/database'

def get_db_folders():
    """Get list of database folders in Spider dataset"""
    if not SPIDER_ROOT.exists():
        logger.warning(f"Spider database directory not found at {SPIDER_ROOT}. Creating it now.")
        SPIDER_ROOT.mkdir(parents=True, exist_ok=True)
        return []
    
    return [d for d in SPIDER_ROOT.iterdir() if d.is_dir()]

# Get all database folders
db_folders = get_db_folders()
logger.info(f"Found {len(db_folders)} databases in Spider dataset")

2025-09-17 11:14:25,587 - INFO - Found 166 databases in Spider dataset


In [3]:
# Initialize processor with project root path
processor = SpiderDataProcessor(project_root)

# Get database statistics
db_stats = processor.get_db_statistics()

# Log database statistics
logger.info("\nSpider Database Statistics:")
logger.info("-" * 40)

total_dbs = len(db_stats)
dbs_with_sqlite = sum(1 for stats in db_stats.values() if stats['has_sqlite'])
dbs_with_schema = sum(1 for stats in db_stats.values() if stats['has_schema'])
total_tables = sum(stats['table_count'] for stats in db_stats.values())

logger.info(f"Total databases: {total_dbs}")
logger.info(f"Databases with SQLite files: {dbs_with_sqlite}")
logger.info(f"Databases with schema files: {dbs_with_schema}")
logger.info(f"Total tables across all databases: {total_tables}")
logger.info(f"Average tables per database: {total_tables/dbs_with_sqlite:.2f}")

# Additional Spider dataset information
logger.info("\nDetailed database statistics have been saved to spider_info.json")
logger.info("You can find it in the data directory")

2025-09-17 11:14:26,860 - INFO - 
Spider Database Statistics:
2025-09-17 11:14:26,861 - INFO - ----------------------------------------
2025-09-17 11:14:26,862 - INFO - Total databases: 166
2025-09-17 11:14:26,862 - INFO - Databases with SQLite files: 166
2025-09-17 11:14:26,863 - INFO - Databases with schema files: 148
2025-09-17 11:14:26,864 - INFO - Total tables across all databases: 876
2025-09-17 11:14:26,865 - INFO - Average tables per database: 5.28
2025-09-17 11:14:26,866 - INFO - 
Detailed database statistics have been saved to spider_info.json
2025-09-17 11:14:26,866 - INFO - You can find it in the data directory
2025-09-17 11:14:26,861 - INFO - ----------------------------------------
2025-09-17 11:14:26,862 - INFO - Total databases: 166
2025-09-17 11:14:26,862 - INFO - Databases with SQLite files: 166
2025-09-17 11:14:26,863 - INFO - Databases with schema files: 148
2025-09-17 11:14:26,864 - INFO - Total tables across all databases: 876
2025-09-17 11:14:26,865 - INFO - Aver

In [4]:
# Process Spider train data
logger.info("\nProcessing Spider Training Data:")
logger.info("-" * 40)

try:
    processor.process_spider_train_data()
except Exception as e:
    logger.error(f"Error processing Spider train data: {str(e)}")
    logger.error("Please check if train_spider.json exists and is accessible")

2025-09-17 11:14:29,421 - INFO - 
Processing Spider Training Data:
2025-09-17 11:14:29,422 - INFO - ----------------------------------------
2025-09-17 11:14:29,422 - INFO - ----------------------------------------


#### 2.1 Prompt Design for Role Assignment

The prompt is designed to:
1. Provide clear context about the task (Role-Based Access Control)
2. Guide the LLM to analyze table schema and relationships
3. Generate appropriate role names and descriptions
4. Consider security implications
5. Maintain consistency across different tables

In [ ]:
# Process databases with role assignment
N_SAMPLES = 12
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
logger.info(f"Starting role assignment process for {N_SAMPLES} databases")

# Initialize parallel role generator
generator = ParallelRoleGenerator(model="deepseek-chat", api_key=DEEPSEEK_API_KEY, n_workers=10)
logger.info(f"Initialized ParallelRoleGenerator with model: deepseek-chat")

# Select random databases to process
test_dbs = random.sample(db_folders, N_SAMPLES)
db_names = [db.name for db in test_dbs]
logger.info(f"Selected databases: {db_names}")

# Process databases in parallel
logger.info("Starting parallel processing of databases")
# Include database paths for table validation
sqlite_paths = {db.name: str(db / f"{db.name}.sqlite") for db in test_dbs}
results = generator.process_databases_parallel(test_dbs, sqlite_paths=sqlite_paths)

# Store results
role_assignments = {}
processed_count = 0
total_roles = 0

for result in results:
    if result and result.get('roles'):
        processed_count += 1
        roles_count = len(result['roles'])
        total_roles += roles_count
        role_assignments[result['database']] = result['roles']
    else:
        logger.error(f"Failed to process one of the databases")

# Prepare metadata
assignments_data = {
    'assignments': role_assignments,
    'metadata': {
        'timestamp': timestamp,
        'total_databases': len(test_dbs),
        'processed_databases': processed_count,
        'total_roles_generated': total_roles
    }
}

# Save results if we have any
if role_assignments:
    output_file = generator.save_assignments_parallel(assignments_data, output_dir, timestamp)

logger.info(f"\nProcess completed:")
logger.info(f"- Databases processed: {processed_count}/{len(test_dbs)}")
logger.info(f"- Total roles generated: {total_roles}")